> gen -> sleep -> update actor -> update_weights(actor 同步到 rollout)/wake_up
- 每 step 先 rollout 生成，然后立刻 sleep_replicas()：ray_trainer.py:1313
- actor 更新后进入 checkpoint_manager.update_weights()：ray_trainer.py:1524, base.py:396
- 关键顺序在 engine_workers.update_weights()：
    - 先 rollout.resume(tags=["weights"])（会触发 vLLM wake_up）engine_workers.py:642
    - 然后才 actor.engine.to("cpu") offload actor engine_workers.py:672
    - 最后再 resume(tags=["kv_cache"]) engine_workers.py:677
- 所以在 wake_up(weights) 时，actor 训练态显存通常还在，rollout 权重又回灌，形成瞬时叠加峰值。
    - 先 resume(weights)，触发 vLLM wake_up，wake_up 内部会 create_and_map 把权重显存重新映射回 GPU
    - 然后才从 actor 取参数并同步到 rollout
    - 最后才把 actor offload 到 CPU
    - 所以中间有一个时间窗：rollout weights 已恢复 且 actor 训练态显存还在，必然短时叠加。

### hybrid colocate

- 共同部署
    - 训练和推理，训练：actor/ref => worker, 推理：vllm server => worker
    - 训练侧在更新完 rollout 权重后，会把 actor model offload 回 CPU，
        - verl/workers/engine_workers.py
            - `self.actor.engine.to("cpu", model=True, optimizer=False, grad=False)`
- rollout 阶段的显存
    - vllm + trainer (grad/optimizer，但是都是按照 fsdp 分片的)

### colocate 模式下的显存分析

- rollout
    - vLLM 是主占用方。update_weights() 结束后会把 rollout 的 kv_cache 再唤醒，见 engine_workers.py:675。
    - trainer 不会变成 0 GiB，只会降成一个“底座”，主要是 optimizer state、grad buffer、CUDA/NCCL context 这些常驻状态。模型参数 shard 本身已经被 offload 到 CPU 了，见 engine_workers.py:671（`self.actor.engine.to("cpu", model=True, optimizer=False, grad=False)`）。
- training
    - 训练开始前会先让 rollout sleep_replicas()，目的是释放 rollout 的权重和 KV cache 显存，见 ray_trainer.py:1313 和 checkpoint_engine/base.py:384。
    - vLLM 在 hybrid 模式下的 sleep(level=2) 会尽量把 weights 和 kv_cache 都睡掉，见 vllm_async_server.py:623。
    - 这时主占用方变成 trainer：actor model shard、optimizer states、grad buffers、forward/backward activations、FSDP 通信 buffer。
    - vLLM 这边通常只剩一个很小的“底座”，比如进程本身、CUDA context、少量 runtime 状态，不应该还保留那种 18GB 级别的大块显存。

```mermaid
sequenceDiagram
      autonumber
      participant Driver as RayPPOTrainer
      participant Rollout as vLLM Rollout
      participant Trainer as Trainer Worker
      participant GPU as Same GPU

      Note over Driver,Trainer: Hybrid colocate: rollout 和 trainer 常驻在同一张物理 GPU 上
      Note over GPU: 显存不是硬切分，而是两个进程动态占用

      Driver->>Rollout: generate_sequences()
      activate Rollout
      Note over Rollout,GPU: Rollout phase<br/>vLLM weights + KV cache + MM encoder activations 占大头
      Note over Trainer,GPU: Trainer 仍保留底座<br/>optimizer state + grad buffer + CUDA/NCCL context
      Rollout-->>Driver: sampled responses
      deactivate Rollout

      Driver->>Rollout: sleep_replicas()
      activate Rollout
      Rollout->>GPU: sleep(level=2)
      Note over Rollout,GPU: 尽量释放 vLLM weights + KV cache
      Rollout-->>Driver: rollout memory reduced
      deactivate Rollout

      Driver->>Trainer: compute_old_log_prob()
      activate Trainer
      Note over Trainer,GPU: Training subphase<br/>actor model shard 重新上 GPU 做前向
      Trainer-->>Driver: old_log_probs
      deactivate Trainer

      Driver->>Trainer: compute_ref_log_prob()
      activate Trainer
      Trainer-->>Driver: ref_log_probs
      deactivate Trainer

      Driver->>Trainer: update_actor()
      activate Trainer
      Note over Trainer,GPU: 训练主相位<br/>model shard + optimizer state + grad + activations
      Trainer-->>Driver: updated actor
      deactivate Trainer

      Driver->>Trainer: update_weights()
      activate Trainer
      Driver->>Rollout: resume(tags=["weights"])
      activate Rollout
      Note over Trainer,Rollout: 切换相位，最容易出现瞬时峰值
      Trainer->>Rollout: stream latest model weights
      Trainer->>Trainer: to("cpu", model=True, optimizer=False, grad=False)
      Note over Trainer,GPU: 只下掉 model shard<br/>optimizer/grad 仍留在 GPU
      Rollout->>GPU: resume(tags=["kv_cache"])
      Note over Rollout,GPU: vLLM 再次拿回 rollout 所需显存
      Rollout-->>Driver: rollout ready
      deactivate Rollout
      deactivate Trainer

      Driver->>Rollout: next generate_sequences()
      activate Rollout
      Note over GPU: 回到稳态 rollout<br/>vLLM 大块 + trainer 底座
      Rollout-->>Driver: next batch
      deactivate Rollout
```

### ray 架构：单控制器 + 多 wg（worker group）

- actor_rollout_wg:
    - compute_log_prob
    - update_actor
- ref_policy_wg
    - compute_ref_log_prob
- critic_wg
    - compute_values
    - update_critic

```python
out = wg.generate_sequences(batch)

# 实际等价于（概念上）
shards = split(batch, world_size)
refs = [worker_i.generate_sequences.remote(shards[i]) for i in range(world_size)]
out = concat(ray.get(refs))
```
- RayWorkerGroup 在初始化时，会扫描 Worker 类中所有带 @register 装饰器的方法，并动态地通过 setattr 将它们挂载到自己身上。
- ActorRolloutRefWorker.generate_sequences
    - verl/workers/rollout/vllm_rollout.py

### Training Step 同步序列                                                                                                                           

1. generate_sequences()     → 所有 DP rank 生成完成                                                                                                             
2. compute_rm_score()       → Reward Model 计算                                                                                                                 
3. compute_log_prob()       → Actor 重新计算 log prob                                                                                                           
4. compute_ref_log_prob()   → Reference Policy 计算                                                                                                             
5. compute_values()         → Critic 计算 Values                                                                                                                
6. update_critic()          → Critic 梯度更新 (NCCL AllReduce)                                                                                                  
7. update_actor()           → Actor 梯度更新 (NCCL AllReduce)      

### metrics

- timing_s: iming_s/gen|old_log_prob|ref|values|update_critic|update_actor|save_checkpoint